# AI Text Humanizer — T5 Fine-Tuning v5 (Stable)

**Author**: Goutham (SISTK)

## v5 Fixes:
| Problem | Fix |
|---|---|
| HC3/PAWS/Quora broken (script-based) | Use `glue/mrpc` + `glue/qqp` (always Parquet, always works) |
| NaN from fp16 + checkpoint tie conflict | `USE_FP16 = False` + `tie_word_embeddings = False` |
| Only 62 pairs | 50 curated + 200+ from GLUE + 50 rule-generated = 300+ pairs |

> Runtime -> Change runtime type -> **T4 GPU**


## Step 1 — Install

In [ ]:
!pip install -q transformers datasets torch sentencepiece accelerate
print('Done.')

## Step 2 — GPU Check

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# v5 FIX: Disable fp16 — it causes NaN when resuming from checkpoint with tie_word_embeddings conflict
USE_FP16 = False
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'fp16   : DISABLED (fp32 for stable training — avoids NaN from checkpoint tie conflict)')

## Step 3 — Rule-Based Engine

In [ ]:
import re, random

VOCAB_MAP = {
    'delve': ['examine', 'investigate', 'explore', 'look at'],
    'delves': ['examines', 'investigates', 'explores'],
    'delving': ['examining', 'investigating', 'exploring'],
    'leverage': ['use', 'utilize', 'apply', 'harness'],
    'leverages': ['uses', 'utilizes', 'applies'],
    'leveraging': ['using', 'utilizing', 'applying'],
    'comprehensive': ['thorough', 'detailed', 'complete'],
    'crucial': ['vital', 'essential', 'key', 'important'],
    'multifaceted': ['complex', 'varied', 'diverse'],
    'underscores': ['highlights', 'emphasizes', 'notes'],
    'underscore': ['highlight', 'emphasize', 'note'],
    'paramount': ['vital', 'critical', 'essential'],
    'meticulous': ['careful', 'thorough', 'detailed'],
    'meticulously': ['carefully', 'thoroughly', 'in detail'],
    'foster': ['promote', 'build', 'support'],
    'fosters': ['promotes', 'builds', 'supports'],
    'pivotal': ['central', 'key', 'major'],
    'seamlessly': ['smoothly', 'easily', 'naturally'],
    'collaborated': ['worked jointly', 'teamed up'],
    'optimization': ['refinement', 'tuning'],
    'achieving': ['delivering', 'securing', 'reaching'],
    'achieved': ['delivered', 'secured', 'reached'],
    'robust': ['strong', 'reliable', 'solid'],
    'innovative': ['creative', 'novel', 'new'],
    'facilitate': ['help', 'support', 'enable'],
    'facilitates': ['helps', 'supports', 'enables'],
    'subsequently': ['then', 'next', 'after that'],
    'thereby': ['thus', 'so', 'as a result'],
    'utilize': ['use', 'apply', 'employ'],
    'utilizes': ['uses', 'applies', 'employs'],
}

BURSTINESS_TRANSFORMS = [
    (r'from,\s*', 'from '),
    (r'\bI final year student from\b', 'currently in my final year at'),
    (r'^\b(Collaborated|Worked jointly|Teamed up|Partnered) on\b', 'I worked on'),
    (r',?\s*\b(achieving|delivering|securing|producing)\s+(a\s+)?(\d+%)\s+(reduction|decrease|drop)\s+in\s+API\s+response\s+latency\b', r' to cut API response latency by \3'),
    (r'\bcore production features\b', 'core features'),
    (r'\bproduction features\b', 'key features'),
    (r'^\b(Achieved|Delivered|Secured|Produced)\b', 'I delivered'),
    (r'^\b(Implemented|Engineered|Developed|Built)\b', 'I built'),
    (r'^\b(Optimized|Refined|Enhanced)\b', 'I optimized'),
    (r'^\b(Leveraged|Utilized|Harnessed)\b', 'I used'),
    (r'^\b(Leveraging|Utilizing)\b', 'Using'),
]

TRANSITION_MAP = {
    'furthermore': 'in addition', 'moreover': 'alongside this',
    'consequently': 'as a result', 'therefore': 'thus',
    'however': 'though', 'in conclusion': 'overall',
    'in summary': 'in short', 'additionally': 'also',
}

def vocabulary_rewrite(text):
    words = text.split()
    result = []
    for word in words:
        m = re.match(r'^([^\w]*)(\w[\w-]*)([^\w]*)$', word)
        if m:
            prefix, clean, suffix = m.groups()
            if clean.lower() in VOCAB_MAP:
                syn = random.choice(VOCAB_MAP[clean.lower()])
                if clean[0].isupper(): syn = syn.capitalize()
                result.append(f'{prefix}{syn}{suffix}')
            else:
                result.append(word)
        else:
            result.append(word)
    return ' '.join(result)

def burstiness_rewrite(text):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    out = []
    for s in sents:
        for p, r in BURSTINESS_TRANSFORMS:
            s = re.sub(p, r, s, flags=re.IGNORECASE)
        out.append(re.sub(r'\s+', ' ', s).strip())
    return ' '.join(out)

def perplexity_rewrite(text):
    for k, v in TRANSITION_MAP.items():
        text = re.sub(rf'\b{k}\b', v, text, flags=re.IGNORECASE)
    return text

def post_process(text):
    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(into|in|at|on|of)\s+\1\b', r'\1', text)
    text = re.sub(r'^By using\b', 'Using', text)
    text = re.sub(r'\bcurrently my\b', 'currently in my', text)
    return re.sub(r'\s+', ' ', text).strip()

def combined_rewrite(text):
    return post_process(perplexity_rewrite(vocabulary_rewrite(burstiness_rewrite(text))))

print('Rule-based engine ready.')

## Step 4 — Load Dataset

**v5 Datasets (all Parquet-based, no script loading):**
- `glue/mrpc` — Microsoft Paraphrase Corpus (3668 pairs, always works)
- `glue/qqp` — Quora Question Pairs via GLUE (parquet version)
- 50 curated domain pairs (resume, academic, tech)
- 50 rule-generated pairs


In [ ]:
from datasets import load_dataset

# 50 curated domain-specific pairs
pairs = [
    {'input': 'Collaborated on database query optimization and schema design, achieving a 25% reduction in API response latency across 10+ core production features.',
     'target': 'I worked on database query optimization and schema design to cut API response latency by 25% across 10+ core features.'},
    {'input': 'Furthermore, it is crucial to delve into the multifaceted structure of transformer models.',
     'target': 'We explored the complex structure of transformer models in detail.'},
    {'input': 'Leveraging comprehensive NLP techniques is paramount for establishing robust baseline metrics.',
     'target': 'Using thorough NLP techniques is key to building strong baseline metrics.'},
    {'input': 'The experiments were meticulously conducted to foster empirical validity across all test conditions.',
     'target': 'We carefully ran the experiments to ensure reliable results across all test conditions.'},
    {'input': 'Implemented a scalable microservices architecture leveraging Docker and Kubernetes for seamless deployment.',
     'target': 'I built a scalable microservices architecture using Docker and Kubernetes for smooth deployment.'},
    {'input': 'Moreover, the proposed framework underscores the pivotal role of data augmentation in model robustness.',
     'target': 'Alongside this, the proposed framework highlights the key role of data augmentation in model robustness.'},
    {'input': 'In conclusion, this comprehensive analysis reveals crucial insights into the multifaceted challenges of AI alignment.',
     'target': 'Overall, this thorough analysis reveals important insights into the complex challenges of AI alignment.'},
    {'input': 'Achieved a 40% improvement in model inference speed through meticulous optimization of the data pipeline.',
     'target': 'I delivered a 40% improvement in model inference speed by carefully optimizing the data pipeline.'},
    {'input': 'Consequently, we must leverage state-of-the-art techniques to address the paramount challenges in this domain.',
     'target': 'As a result, we should use modern techniques to tackle the key challenges in this area.'},
    {'input': 'Engineered a comprehensive RESTful API that seamlessly integrates with existing enterprise infrastructure.',
     'target': 'I built a thorough RESTful API that smoothly integrates with existing enterprise infrastructure.'},
    {'input': 'This multifaceted approach underscores the paramount importance of iterative testing methodologies.',
     'target': 'This complex approach highlights how important iterative testing really is.'},
    {'input': 'Meticulously documented all processes to facilitate seamless onboarding of new team members.',
     'target': 'I carefully documented all processes to help new team members get up to speed quickly.'},
    {'input': 'Leveraging state-of-the-art transformer architectures is pivotal for achieving superior performance benchmarks.',
     'target': 'Using the latest transformer architectures is key for reaching top performance.'},
    {'input': 'Furthermore, meticulous attention to detail is crucial for ensuring comprehensive test coverage.',
     'target': 'In addition, careful attention to detail is essential for thorough test coverage.'},
    {'input': 'The proposed solution seamlessly integrates with the existing codebase, thereby minimizing disruption.',
     'target': 'The solution fits smoothly into the existing codebase, so disruption stays low.'},
    {'input': 'Subsequently, the team utilized advanced machine learning algorithms to optimize the recommendation engine.',
     'target': 'After that, the team used machine learning algorithms to fine-tune the recommendation engine.'},
    {'input': 'This innovative approach fosters a robust and scalable framework for real-time data processing.',
     'target': 'This new approach builds a strong, scalable framework for processing data in real time.'},
    {'input': 'Achieved a paramount reduction in computational overhead through meticulous code refactoring.',
     'target': 'I delivered a key reduction in computational overhead by carefully refactoring the code.'},
    {'input': 'Collaborated with cross-functional teams to implement a robust CI/CD pipeline, achieving a 50% reduction in deployment time.',
     'target': 'I worked with cross-functional teams to build a strong CI/CD pipeline, cutting deployment time by 50%.'},
    {'input': 'Leveraging distributed training frameworks is paramount for scaling model training to billions of parameters.',
     'target': 'Using distributed training frameworks is essential for scaling model training to billions of parameters.'},
    {'input': 'Furthermore, it is pivotal to foster a culture of continuous integration to maintain code quality.',
     'target': 'In addition, it is key to build a culture of continuous integration to maintain code quality.'},
    {'input': 'Optimized the database schema to seamlessly handle concurrent read and write operations at scale.',
     'target': 'I optimized the database schema to smoothly handle concurrent reads and writes at scale.'},
    {'input': 'The comprehensive benchmark suite underscores the paramount importance of reproducibility in AI research.',
     'target': 'The thorough benchmark suite highlights how critical reproducibility is in AI research.'},
    {'input': 'Implemented robust caching mechanisms that seamlessly reduce API response time by 60%.',
     'target': 'I built strong caching mechanisms that smoothly cut API response time by 60%.'},
    {'input': 'It is paramount to leverage comprehensive data validation pipelines to maintain data integrity.',
     'target': 'It is essential to use thorough data validation pipelines to maintain data integrity.'},
    {'input': 'The novel framework meticulously addresses the multifaceted challenges of low-resource language translation.',
     'target': 'The new framework carefully addresses the complex challenges of low-resource language translation.'},
    {'input': 'Collaborated on designing a seamless user onboarding experience, achieving a 35% increase in activation rate.',
     'target': 'I worked on designing a smooth user onboarding experience, securing a 35% increase in activation rate.'},
    {'input': 'The system leverages a comprehensive ensemble of models to foster more robust predictions.',
     'target': 'The system uses a thorough ensemble of models to build more reliable predictions.'},
    {'input': 'In summary, these findings underscore the pivotal role of data quality in model generalization.',
     'target': 'In short, these findings highlight the key role of data quality in model generalization.'},
    {'input': 'The team utilized meticulous hyperparameter tuning to achieve state-of-the-art results on the benchmark.',
     'target': 'The team used careful hyperparameter tuning to reach top results on the benchmark.'},
    {'input': 'Moreover, it is crucial to foster interdisciplinary collaboration to address multifaceted societal challenges.',
     'target': 'Alongside this, it is important to support collaboration to address complex societal challenges.'},
    {'input': 'Engineered a seamless data ingestion pipeline leveraging Apache Kafka for real-time event streaming.',
     'target': 'I built a smooth data ingestion pipeline using Apache Kafka for real-time event streaming.'},
    {'input': 'Implemented a comprehensive A/B testing framework to meticulously evaluate feature impact on user retention.',
     'target': 'I built a thorough A/B testing framework to carefully measure feature impact on user retention.'},
    {'input': 'Consequently, meticulous code review processes were implemented to foster higher software quality standards.',
     'target': 'As a result, careful code review processes were put in place to support higher quality standards.'},
    {'input': 'Leveraging transfer learning is paramount for achieving competitive performance with limited labeled data.',
     'target': 'Using transfer learning is essential for achieving good performance with limited labeled data.'},
    {'input': 'Furthermore, the pivotal contribution of this work lies in its novel approach to few-shot learning.',
     'target': 'In addition, the key contribution of this work is its new approach to few-shot learning.'},
    {'input': 'Optimized query performance by leveraging comprehensive database indexing strategies, achieving a 45% speedup.',
     'target': 'I optimized query performance by using thorough database indexing strategies, reaching a 45% speedup.'},
    {'input': 'The proposed framework seamlessly combines rule-based and neural approaches to foster robust NLP solutions.',
     'target': 'The proposed framework smoothly combines rule-based and neural approaches to build strong NLP solutions.'},
    {'input': 'It is crucial to delve into the comprehensive literature to understand the multifaceted landscape of AI safety.',
     'target': 'It is important to explore the thorough literature to understand the complex landscape of AI safety.'},
    {'input': 'The meticulously designed evaluation protocol ensures comprehensive and unbiased model assessment.',
     'target': 'The carefully designed evaluation protocol ensures thorough and unbiased model assessment.'},
    {'input': 'Achieved a 30% reduction in model training time by leveraging optimized data loading pipelines.',
     'target': 'I cut model training time by 30% by using optimized data loading pipelines.'},
    {'input': 'The multifaceted nature of deep learning necessitates a comprehensive understanding of gradient descent.',
     'target': 'The complex nature of deep learning requires a thorough understanding of gradient descent.'},
    {'input': 'Implemented seamless OAuth 2.0 integration to foster secure and robust user authentication flows.',
     'target': 'I built smooth OAuth 2.0 integration to support secure and strong user authentication flows.'},
    {'input': 'The paramount challenge in NLP is to leverage comprehensive datasets to foster cross-lingual robustness.',
     'target': 'The key challenge in NLP is to use thorough datasets to build cross-lingual robustness.'},
    {'input': 'Subsequently leveraging containerized microservices fostered a paramount reduction in deployment complexity.',
     'target': 'After that, using containerized microservices helped achieve a key reduction in deployment complexity.'},
    {'input': 'Meticulously profiled the application to identify and resolve critical bottlenecks in the data pipeline.',
     'target': 'I carefully profiled the application to find and fix critical bottlenecks in the data pipeline.'},
    {'input': 'The comprehensive research agenda delves into multifaceted aspects of responsible AI development.',
     'target': 'The thorough research agenda explores complex aspects of responsible AI development.'},
    {'input': 'Leveraged seamless WebSocket integration to foster real-time collaborative features across all platforms.',
     'target': 'I used smooth WebSocket integration to support real-time collaborative features across all platforms.'},
    {'input': 'It is crucial to foster meticulous data governance practices to ensure comprehensive regulatory compliance.',
     'target': 'It is important to support careful data governance practices to ensure thorough regulatory compliance.'},
    {'input': 'The robust system architecture seamlessly handles millions of concurrent users with paramount reliability.',
     'target': 'The strong system architecture smoothly handles millions of concurrent users with vital reliability.'},
    {'input': 'Collaborated on implementing comprehensive end-to-end encryption to foster paramount data security.',
     'target': 'I worked on implementing thorough end-to-end encryption to support essential data security.'},
]

print(f'Curated pairs: {len(pairs)}')

# --- glue/mrpc: Microsoft Paraphrase Corpus (Parquet, always works) ---
try:
    mrpc = load_dataset('glue', 'mrpc', split='train')
    before = len(pairs)
    for item in mrpc:
        if item['label'] == 1:  # 1 = true paraphrase
            s1, s2 = item['sentence1'].strip(), item['sentence2'].strip()
            if 20 < len(s1) < 300 and 20 < len(s2) < 300:
                pairs.append({'input': s1, 'target': s2})
    print(f'glue/mrpc: +{len(pairs)-before} pairs. Total: {len(pairs)}')
except Exception as e:
    print(f'glue/mrpc FAILED: {e}')

# --- glue/qqp: Quora Question Pairs via GLUE (Parquet version, no script) ---
try:
    qqp = load_dataset('glue', 'qqp', split='train[:800]')
    before = len(pairs)
    for item in qqp:
        if item['label'] == 1:  # 1 = duplicate/paraphrase
            q1, q2 = item['question1'].strip(), item['question2'].strip()
            if 15 < len(q1) < 250 and 15 < len(q2) < 250:
                pairs.append({'input': q1, 'target': q2})
    print(f'glue/qqp : +{len(pairs)-before} pairs. Total: {len(pairs)}')
except Exception as e:
    print(f'glue/qqp FAILED: {e}')

# --- dmitva: AI vs Human text (Parquet) ---
try:
    dv = load_dataset('dmitva/human_ai_generated_text', split='train[:300]')
    before = len(pairs)
    for item in dv:
        ai  = (item.get('ai_text') or item.get('generated_text', '')).strip()
        hum = (item.get('human_text') or item.get('text', '')).strip()
        if ai and hum and 20 < len(ai) < 400 and 20 < len(hum) < 400:
            pairs.append({'input': ai, 'target': hum})
    print(f'dmitva   : +{len(pairs)-before} pairs. Total: {len(pairs)}')
except Exception as e:
    print(f'dmitva FAILED: {e}')

# --- Rule-generated pairs ---
extra_ai = [
    'Moreover, it is crucial to leverage comprehensive data preprocessing pipelines to foster model generalization.',
    'Implemented robust error handling mechanisms that seamlessly integrate with the existing codebase.',
    'Consequently, this multifaceted approach underscores the paramount importance of iterative testing.',
    'The comprehensive evaluation framework delves into the multifaceted nuances of language model performance.',
    'In summary, these comprehensive experiments validate the crucial role of attention mechanisms in NLP tasks.',
    'Furthermore, meticulous documentation was provided to facilitate seamless onboarding of new team members.',
    'The innovative solution seamlessly integrates with existing systems, thereby reducing operational overhead.',
    'Subsequently, the team utilized robust machine learning algorithms to optimize the recommendation pipeline.',
    'Leveraging a comprehensive suite of evaluation metrics is paramount for achieving robust model assessment.',
    'This multifaceted research endeavors to delve into crucial nuances of cross-lingual transfer learning.',
    'Furthermore, the meticulous design of the experiment fosters comprehensive validation of the hypothesis.',
    'Consequently, the team leveraged a robust distributed system to seamlessly handle peak traffic loads.',
    'Meticulously optimizing the data pipeline fostered a paramount improvement in model training throughput.',
    'The team leveraged comprehensive observability tools to foster seamless system reliability and uptime.',
    'Furthermore, it is crucial to delve into the multifaceted implications of federated learning for privacy.',
    'Implemented a robust feature flagging system to facilitate seamless progressive rollout of new functionality.',
    'The meticulous analysis underscores the pivotal role of tokenization in multilingual language model performance.',
    'Consequently, leveraging comprehensive user feedback loops is paramount for fostering product-market fit.',
]
before = len(pairs)
for ai_text in extra_ai:
    human_text = combined_rewrite(ai_text)
    if ai_text.lower() != human_text.lower():
        pairs.append({'input': ai_text, 'target': human_text})
print(f'Rule-gen : +{len(pairs)-before} pairs. Total: {len(pairs)}')

random.shuffle(pairs)
print(f'\nFINAL dataset: {len(pairs)} pairs')
print(f'Sample IN : {pairs[0]["input"][:80]}...')
print(f'Sample OUT: {pairs[0]["target"][:80]}...')

## Step 5 — Tokenize & Build DataLoader

In [ ]:
import os, math
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

MODEL_NAME     = 'google/flan-t5-base'
CHECKPOINT_DIR = '/content/t5-humanizer-v1'
OUTPUT_DIR     = '/content/t5-humanizer-v1'
MAX_LENGTH     = 128
BATCH_SIZE     = 8
EPOCHS         = 10
LR             = 5e-5
MAX_GRAD_NORM  = 1.0

tok_src = CHECKPOINT_DIR if os.path.exists(os.path.join(CHECKPOINT_DIR, 'config.json')) else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tok_src)
print(f'Tokenizer loaded from: {tok_src}')

class HumanizerDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=MAX_LENGTH):
        self.pairs, self.tokenizer, self.max_length = pairs, tokenizer, max_length
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        p = self.pairs[idx]
        enc = self.tokenizer(f"humanize: {p['input']}", padding='max_length',
                             truncation=True, max_length=self.max_length, return_tensors='pt')
        dec = self.tokenizer(p['target'], padding='max_length',
                             truncation=True, max_length=self.max_length, return_tensors='pt')
        labels = dec['input_ids'].squeeze(0).clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': labels}

dataset    = HumanizerDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
total_steps  = EPOCHS * len(dataloader)
warmup_steps = max(1, int(0.05 * total_steps))
print(f'Dataset: {len(dataset)} samples | {len(dataloader)} batches/epoch | {total_steps} steps | {warmup_steps} warmup')

## Step 6 — Train (fp32, stable, NaN-safe)

**v5 NaN fixes:**
- `USE_FP16 = False` — fp32 avoids all overflow issues
- `model.config.tie_word_embeddings = False` — silences warning, prevents weight conflict
- Standard training loop — no GradScaler complexity


In [ ]:
# Load model
ckpt_ok = os.path.exists(os.path.join(CHECKPOINT_DIR, 'config.json'))
src = CHECKPOINT_DIR if ckpt_ok else MODEL_NAME
print(f'Loading model from: {src}')
model = AutoModelForSeq2SeqLM.from_pretrained(src).to(device)

# v5 FIX: suppress tie_word_embeddings conflict that caused NaN in fp16
model.config.tie_word_embeddings = False

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
if ckpt_ok:
    print('Resuming from v3 checkpoint (loss 2.418). Training in fp32.')

# Optimizer
no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = AdamW(params, lr=LR)

# Cosine LR with warmup
def cosine_warmup(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(1e-7, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda=cosine_warmup)

print(f'Config: epochs={EPOCHS}, lr={LR}, clip={MAX_GRAD_NORM}, fp16=OFF')
print('=' * 65)

model.train()
best_loss    = float('inf')
loss_history = []
global_step  = 0

for epoch in range(EPOCHS):
    total_loss = 0.0
    nan_steps  = 0

    for step, batch in enumerate(dataloader):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        # fp32 forward — stable, no NaN
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss

        if torch.isnan(loss) or torch.isinf(loss):
            nan_steps += 1
            print(f'  WARNING: NaN/Inf at step {step+1}, skipping.')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()   # optimizer FIRST
        scheduler.step()   # scheduler AFTER

        global_step += 1
        total_loss  += loss.item()

        if (step + 1) % 10 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f'  Ep {epoch+1}/{EPOCHS} | Step {step+1}/{len(dataloader)} | Loss: {loss.item():.4f} | LR: {lr_now:.2e}')

    valid = len(dataloader) - nan_steps
    avg   = total_loss / max(1, valid)
    loss_history.append(avg)

    tag = ''
    if nan_steps: tag += f' [{nan_steps} NaN skipped]'
    if avg < best_loss:
        best_loss = avg
        tag += ' *BEST*'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
    print(f'Epoch {epoch+1:2d} | Avg Loss: {avg:.4f}{tag}')

print('=' * 65)
print(f'Training complete! Best loss: {best_loss:.4f}')
print(f'History: {[round(l,4) for l in loss_history]}')

## Step 7 — Test

In [ ]:
inf_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)
inf_tok   = AutoTokenizer.from_pretrained(OUTPUT_DIR)
inf_model.eval()

def humanize(text):
    rule = combined_rewrite(text)
    inp  = inf_tok(f'humanize: {rule}', return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = inf_model.generate(**inp, max_length=128, num_beams=4,
                                  temperature=0.8, top_p=0.92,
                                  do_sample=True, no_repeat_ngram_size=2)
    return post_process(inf_tok.decode(out[0], skip_special_tokens=True))

tests = [
    'I am Goutham from, Tirupati district and I final year student from SISTK. Collaborated on database query optimization and schema design, achieving a 25% reduction in API response latency across 10+ core production features.',
    'Furthermore, it is crucial to delve into the multifaceted challenges of AI alignment to foster responsible development.',
    'Leveraging comprehensive transformer architectures is paramount for achieving seamless language understanding at scale.',
]
print('=== Test Results ===')
for i, t in enumerate(tests):
    print(f'[{i+1}] IN : {t}')
    print(f'    OUT: {humanize(t)}')
    print()

## Step 8 — Download

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/t5-humanizer-v1', 'zip', OUTPUT_DIR)
files.download('/content/t5-humanizer-v1.zip')
print('Done. Place extracted files in: d:\\AI_Text_Checker\\models\\t5-humanizer-v1\\')